# Notebook 5 - Transaction Engine

this one took me a while to get right. the idea is simple - compare this quarter's holdings to last quarter's holdings and figure out what changed.

technically its a full outer join on cusip and put_call between consecutive quarters:
- position exists this quarter but not last quarter = NEW POSITION  
- position existed last quarter but not this quarter = FULL EXIT
- shares went up = BUY
- shares went down = SELL
- shares basically the same = HOLD

key thing - we compare SHARES not dollar values. dollar values change every quarter just from price movements even if the manager didnt do anything. shares only change when there is an actual transaction.

there is a small tolerance for rounding too, less than 0.5% share change we just call it HOLD.

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
from src.transactions import build_transactions

tx = build_transactions()
tx.head(10)

In [ ]:
# summary of actions per quarter
tx.pivot_table(index="quarter", columns="action", values="cusip", aggfunc="count", fill_value=0)

In [ ]:
# biggest moves in the most recent quarter
latest = tx["quarter"].max()
(tx[(tx["quarter"] == latest) & (tx["action"] != "HOLD")]
   .assign(abs_change=lambda d: d["value_change_usd"].abs())
   .sort_values("abs_change", ascending=False)
   [["issuer", "cusip", "action", "share_change", "value_change_usd"]]
   .head(15))